In [ ]:
import os
import re

from dotenv import load_dotenv
_ = load_dotenv()

TPCDS_TOOLS_DIR = os.getenv("TPCDS_TOOLS_DIR")
DATA_DIR = os.path.join(TPCDS_TOOLS_DIR, "data")

print(f"TPCDS_TOOLS_DIR: {TPCDS_TOOLS_DIR}")
print(f"DATA_DIR: {DATA_DIR}")

In [ ]:
print(os.listdir(DATA_DIR))

## **Starting Point: SparkSession**
The entry point into all functionality in Spark is the `SparkSession` class. To create a basic `SparkSession`, just use `SparkSession.builder`.

Source: [Spark SQL Guide - Getting started](https://spark.apache.org/docs/latest/sql-getting-started.html)

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("tpcds") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
spark.version

- [Spark SQL - SQL Syntax](https://spark.apache.org/docs/latest/sql-ref-syntax.html)
- [Spark SQL - Data Types](https://spark.apache.org/docs/latest/sql-ref-datatypes.html)

In [ ]:
path = "/home/basbenja/Documents/DEAI/data-warehouses/project/part1/tpcds-tools/tpcds.sql"
with open(path, "r") as f:
    sql = f.read()
    # This will throw an error becuase the SQL file contains multiple statements,
    # and spark.sql only allows one statement at a time.
    try:
        spark.sql(sql)
    except Exception as e:
        print(f"Error executing SQL: {e}")

In [ ]:
try:
    spark.sql("""create table dbgen_version
    (
        dv_version                varchar(16)                   ,
        dv_create_date            date                          ,
        dv_create_time            time                          ,
        dv_cmdline_args           varchar(200)
    );""")
except Exception as e:
    print(f"Error creating table dbgen_version: {e}")

In [ ]:
df = spark.read.csv(
    path=os.path.join(DATA_DIR, "warehouse.dat"),
    sep='|'
)
print(type(df))
# We see that the data has been loaded correctly but it has no schema (all columns
# are unnamed)
df.show()

Now, we will try load the schema of this data and associate it.

In [ ]:
sql_schema_path = os.path.join(TPCDS_TOOLS_DIR, "tpcds.sql")
with open(sql_schema_path, "r") as f:
    sql_schema = f.read()

# We remove the comments (lines starting with --)
no_comments = "\n".join(
    line for line in sql_schema.splitlines() if not line.strip().startswith("--")
)
# We remove the primary key constraints (lines containing "primary key"). Spark
# does not support them
# drop primary key / foreign key constraint lines
no_constraints = "\n".join(
    line for line in no_comments.splitlines()
    if not re.match(r"\s*(primary|foreign)\s+key", line, re.IGNORECASE)
)
# clean up the trailing comma left behind
no_constraints = re.sub(r",\s*\)", ")", no_constraints)

schemas = [
    table.strip() for table in no_constraints.split(";") if table.strip()
]

# Map each table name to its full "create table" statement
table_name_re = re.compile(r"create table (\w+)", re.IGNORECASE)
table_schemas = {
    table_name_re.search(s).group(1): s
    for s in schemas
}

for name, s in table_schemas.items():
    print(name)
    print(s)
    print()

In [ ]:
for k, v in table_schemas.items():
    data_path = os.path.join(DATA_DIR, f"{k}.dat")
    if not os.path.exists(data_path):
        print(f"Data file for table {k} does not exist at {data_path}")
        continue

    # Number of columns defined in the schema (one column definition per line)
    body = v[v.index("(") + 1 : v.rindex(")")]
    schema_cols = len([line for line in body.splitlines() if line.strip()])

    # Number of columns in the .dat file, based on its first row. TPC-DS .dat
    # files end every row with a trailing delimiter, so split() leaves one
    # extra empty field at the end that we discard.
    with open(data_path, "r") as f:
        first_line = f.readline().rstrip("\n")
    data_cols = len(first_line.split("|")) - 1

    status = "OK" if schema_cols == data_cols else "MISMATCH"
    print(f"{k}: schema={schema_cols}, data={data_cols} [{status}]")

In [ ]:
try:
    spark.sql(f"""create table warehouse
    (
        w_warehouse_sk            integer               not null,
        w_warehouse_id            char(16)              not null,
        w_warehouse_name          varchar(20)                   ,
        w_warehouse_sq_ft         integer                       ,
        w_street_number           char(10)                      ,
        w_street_name             varchar(60)                   ,
        w_street_type             char(15)                      ,
        w_suite_number            char(10)                      ,
        w_city                    varchar(60)                   ,
        w_county                  varchar(30)                   ,
        w_state                   char(2)                       ,
        w_zip                     char(10)                      ,
        w_country                 varchar(20)                   ,
        w_gmt_offset              decimal(5,2)
    )
    USING csv
    OPTIONS (
        path '{os.path.join(DATA_DIR, "warehouse.dat")}',
        header "false",
        delimiter "|"
    );""")
except Exception as e:
    print(f"Error creating table warehouse: {e}")

In [ ]:
spark.conf.get("spark.sql.sources.default")

In [ ]:
spark.sql(""" show tables; """).show()